# GH Archive

_Part of the **Decay Before Archival** project — see `README.md` for full project context, setup instructions, and links to the other notebooks (`deps_dev.ipynb`, `scorecard.ipynb`, `other_sources.ipynb`)._

In [ ]:
from google.cloud import bigquery
from decay_before_archival import get_client

client = get_client()

## Explore: GH Archive

GH Archive tables are sharded by date (`githubarchive.day.YYYYMMDD`). Peek one day's schema and sample rather than scanning the full history.

**Planned analysis:** commit/PR/issue events over a trailing window to derive commit frequency, time-to-resolve, and contributor turnover where the GitHub API alone would be rate-limited or missing history.

In [ ]:
# Picked one day's shard (`githubarchive.day.20250101`) and peek its columns (INFORMATION_SCHEMA.COLUMNS).
shard_date = "20250101"

tables_query = """
SELECT table_name
FROM `githubarchive.day.INFORMATION_SCHEMA.TABLES`
WHERE table_name = @shard_date
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("shard_date", "STRING", shard_date)
    ],
    dry_run=True,
    use_query_cache=False,
)

# Dry run to check the estimated bytes
dry_run = client.query(tables_query, job_config=job_config)
estimated_bytes = dry_run.total_bytes_processed or 0

# Print the dry run results
print(f"Metadata query: {estimated_bytes / 1e6:.2f} MB")

In [ ]:
# Picked one day's shard (`githubarchive.day.20250101`) and peek its columns (INFORMATION_SCHEMA.COLUMNS).
columns_query = """
SELECT column_name, data_type
FROM `githubarchive.day.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = @shard_date
ORDER BY ordinal_position
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("shard_date", "STRING", shard_date)
    ],
    dry_run=False,
    use_query_cache=False,
)

dry_run = client.query(columns_query, job_config=job_config)
estimated_bytes = dry_run.total_bytes_processed or 0
print(f"Metadata query: {estimated_bytes / 1e6:.2f} MB")

client.query(columns_query, job_config=job_config).to_dataframe()

In [ ]:
# Pull a small LIMITed sample from that same day's shard (dry-run first).
sample_query = f"""
SELECT created_at, type, repo, actor
FROM `githubarchive.day.{shard_date}`
LIMIT 20
"""

dry_run = client.query(
    sample_query,
    job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False),
)
estimated_bytes = dry_run.total_bytes_processed or 0
print(f"Sample query: {estimated_bytes / 1e6:.2f} MB")

# Run only after reviewing the estimate
sample = client.query(sample_query).to_dataframe()
sample.head()

**Notes:**

With the sampled shard, it contains multiple event types which includes `PushEvent`, `IssuesEvent`, `DeleteEvent`, and `CreateEvent`, which while useful can make having one row per repository difficult for the table. Since repo and actor are stored as STRUCT, this means that their information is nested inside. Another thing to keep in mind is that a 20 row sample query was estimated to be 909.63 MB. This is why it is important to use dry runs before running a broader query.

## Feature Window & Cutoff


In [ ]:
repo_query = """
WITH repo_events AS (
  SELECT
    repo.id AS repo_id,

    ARRAY_AGG(
      IF(DATE(created_at) < DATE '2025-01-01', repo.name, NULL)
      IGNORE NULLS
      ORDER BY created_at DESC
      LIMIT 1
    )[SAFE_OFFSET(0)] AS repo_full_name,

    COUNTIF(
      DATE(created_at) >= DATE '2024-01-01'
      AND DATE(created_at) < DATE '2024-07-01'
      AND type = 'PushEvent'
    ) AS push_events_h1_2024,

    COUNTIF(
      DATE(created_at) >= DATE '2024-07-01'
      AND DATE(created_at) < DATE '2025-01-01'
      AND type = 'PushEvent'
    ) AS push_events_h2_2024,

    COUNTIF(
      DATE(created_at) >= DATE '2024-01-01'
      AND DATE(created_at) < DATE '2025-01-01'
      AND type = 'PushEvent'
    ) AS push_events_2024,

    MAX(IF(
      DATE(created_at) >= DATE '2024-01-01'
      AND DATE(created_at) < DATE '2025-01-01'
      AND type = 'PushEvent',
      DATE(created_at),
      NULL
    )) AS last_push_date_2024,

    COUNTIF(
      DATE(created_at) >= DATE '2024-01-01'
      AND DATE(created_at) < DATE '2025-01-01'
      AND type = 'IssuesEvent'
    ) AS issue_events_2024,

    COUNTIF(
      DATE(created_at) >= DATE '2024-01-01'
      AND DATE(created_at) < DATE '2025-01-01'
      AND type = 'IssueCommentEvent'
    ) AS issue_comment_events_2024,

    COUNTIF(
      DATE(created_at) >= DATE '2024-01-01'
      AND DATE(created_at) < DATE '2025-01-01'
      AND type = 'PullRequestEvent'
    ) AS pull_request_events_2024,

    COUNTIF(
      DATE(created_at) >= DATE '2024-01-01'
      AND DATE(created_at) < DATE '2025-01-01'
      AND type IN ('PullRequestReviewEvent', 'PullRequestReviewCommentEvent')
    ) AS review_events_2024,

    COUNTIF(
      DATE(created_at) >= DATE '2024-01-01'
      AND DATE(created_at) < DATE '2025-01-01'
      AND type = 'ReleaseEvent'
    ) AS release_events_2024,

    MAX(IF(
      DATE(created_at) >= DATE '2024-01-01'
      AND DATE(created_at) < DATE '2025-01-01'
      AND type = 'ReleaseEvent',
      DATE(created_at),
      NULL
    )) AS last_release_event_date_2024,

    COUNTIF(
      DATE(created_at) >= DATE '2024-01-01'
      AND DATE(created_at) < DATE '2025-01-01'
      AND type = 'WatchEvent'
    ) AS stars_received_2024,

    COUNTIF(
      DATE(created_at) >= DATE '2024-01-01'
      AND DATE(created_at) < DATE '2025-01-01'
      AND type = 'ForkEvent'
    ) AS fork_events_2024,

    COUNT(DISTINCT IF(
      DATE(created_at) >= DATE '2024-01-01'
      AND DATE(created_at) < DATE '2025-01-01'
      AND type IN (
        'PushEvent', 'IssuesEvent', 'IssueCommentEvent',
        'PullRequestEvent', 'PullRequestReviewEvent',
        'PullRequestReviewCommentEvent', 'ReleaseEvent'
      ),
      actor.id,
      NULL
    )) AS distinct_actors_2024,

    COUNT(DISTINCT IF(
      DATE(created_at) >= DATE '2024-01-01'
      AND DATE(created_at) < DATE '2025-01-01'
      AND type IN (
        'PushEvent', 'IssuesEvent', 'IssueCommentEvent',
        'PullRequestEvent', 'PullRequestReviewEvent',
        'PullRequestReviewCommentEvent', 'ReleaseEvent'
      ),
      DATE_TRUNC(DATE(created_at), MONTH),
      NULL
    )) AS active_months_2024,

    COUNT(DISTINCT IF(
      DATE(created_at) >= DATE '2024-01-01'
      AND DATE(created_at) < DATE '2025-01-01'
      AND type IN (
        'PushEvent', 'IssuesEvent', 'IssueCommentEvent',
        'PullRequestEvent', 'PullRequestReviewEvent',
        'PullRequestReviewCommentEvent', 'ReleaseEvent'
      ),
      DATE(created_at),
      NULL
    )) AS active_days_2024,

    COUNTIF(
      DATE(created_at) >= DATE '2024-01-01'
      AND DATE(created_at) < DATE '2024-07-01'
      AND type IN (
        'PushEvent', 'IssuesEvent', 'IssueCommentEvent',
        'PullRequestEvent', 'PullRequestReviewEvent',
        'PullRequestReviewCommentEvent', 'ReleaseEvent'
      )
    ) AS qualifying_events_h1_2024,

    COUNTIF(
      DATE(created_at) >= DATE '2024-07-01'
      AND DATE(created_at) < DATE '2025-01-01'
      AND type IN (
        'PushEvent', 'IssuesEvent', 'IssueCommentEvent',
        'PullRequestEvent', 'PullRequestReviewEvent',
        'PullRequestReviewCommentEvent', 'ReleaseEvent'
      )
    ) AS qualifying_events_h2_2024,

    COUNTIF(
      DATE(created_at) >= DATE '2024-01-01'
      AND DATE(created_at) < DATE '2025-01-01'
      AND type IN (
        'PushEvent', 'IssuesEvent', 'IssueCommentEvent',
        'PullRequestEvent', 'PullRequestReviewEvent',
        'PullRequestReviewCommentEvent', 'ReleaseEvent'
      )
    ) AS qualifying_events_2024,

    COUNTIF(
      DATE(created_at) >= DATE '2025-01-01'
      AND DATE(created_at) < DATE '2026-01-01'
      AND type IN (
        'PushEvent', 'IssuesEvent', 'IssueCommentEvent',
        'PullRequestEvent', 'PullRequestReviewEvent',
        'PullRequestReviewCommentEvent', 'ReleaseEvent'
      )
    ) AS qualifying_events_2025

  FROM `githubarchive.day.20*`
  WHERE _TABLE_SUFFIX BETWEEN '240101' AND '251231'
    AND DATE(created_at) >= DATE '2024-01-01'
    AND DATE(created_at) < DATE '2026-01-01'
    AND repo.id IS NOT NULL
  GROUP BY repo_id
)

SELECT
  repo_id,
  repo_full_name,
  DATE '2025-01-01' AS cutoff_date,
  push_events_h1_2024,
  push_events_h2_2024,
  push_events_2024,
  DATE_DIFF(DATE '2025-01-01', last_push_date_2024, DAY)
    AS days_since_last_push_at_cutoff,
  issue_events_2024,
  issue_comment_events_2024,
  pull_request_events_2024,
  review_events_2024,
  release_events_2024,
  DATE_DIFF(DATE '2025-01-01', last_release_event_date_2024, DAY)
    AS days_since_last_release_event_at_cutoff,
  stars_received_2024,
  fork_events_2024,
  distinct_actors_2024,
  active_months_2024,
  active_days_2024,
  qualifying_events_h1_2024,
  qualifying_events_h2_2024,
  SAFE_DIVIDE(
    qualifying_events_h2_2024 - qualifying_events_h1_2024,
    qualifying_events_h1_2024
  ) AS qualifying_event_change_h2_vs_h1,
  IF(qualifying_events_2025 = 0, 1, 0) AS inactive_next_365d
FROM repo_events
WHERE qualifying_events_2024 > 0
"""

dry_run_config = bigquery.QueryJobConfig(
    dry_run=True,
    use_query_cache=False,
)

dry_run = client.query(repo_query, job_config=dry_run_config)
estimated_bytes = dry_run.total_bytes_processed or 0

print(f"Estimated scan: {estimated_bytes / 1e9:.2f} GB")
print("Actual Run: 200.1 GB")

**Notes:**

This query aggregates events to one row per `repo_id`, with 2024 feature window and `2025-01-01` cutoff, with 2025 qualifying events used to construct the target. While the query dry-run estimated 214.85 GB, the query was run in Google Cloud and was 200.1 GB. Afterwards, it was saved as a BigQuery table and as a smaller CSV file. I was not able add `payload` parsing since during the dry-run estimate, it would have risen to 9.62 TB which would have surpasssed the free 1TB BigQuery limit. 
Another thing to add is that it includes only repositories that have at least one qualifiying maintenance event in 2024.

## Load Data from CSV

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../data/gh_archive_results.csv",
    dtype={"repo_id": "string"},
    low_memory=False,
)

# Display the first 5 rows from gh_archive_results.csv
df.head()

In [ ]:
# Display the column types and data types
df.info()

In [ ]:
# Display the descriptive statistics for each of the columns
df.describe()

## Handling Missing Data

In [ ]:
# Count the missing values in each column
df.isna().sum()

In [ ]:
# 
print("Shape:", df.shape)
print("Duplicate exact rows:", df.duplicated().sum())
print("Duplicate repo/cutoff keys:", df.duplicated(["repo_id", "cutoff_date"]).sum())
print("Missing values by column:")
missing_audit = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
}).sort_values("missing_count", ascending=False)
missing_audit

**Notes:**

Since I downloaded and added the CSV file, I can start to preprocess the data without using the BigQuery table so that I can better understand the dataset. Since the exported dataset has 62,627 repository-cutoff rows, it should be easier to preprocess compared to 54,569,092 rows. The dataset also has not duplicate rows or duplicate `(repo_id, cutoff_date)` keys. There is a bit of an imbalance for the dataset, with there being more repos that are labeled `inactive_next_365d = 1`, which means we cannot rely on that metric to know if a repo was abandoned or archived. 

In [ ]:
# Remove only byte-for-byte duplicate rows. Do not collapse conflicting repo/cutoff rows.
gh_clean = df.drop_duplicates().copy()

gh_clean["repo_id"] = gh_clean["repo_id"].astype("string").str.strip()
gh_clean["repo_full_name"] = (
    gh_clean["repo_full_name"].astype("string").str.strip().str.lower()
)
gh_clean["cutoff_date"] = pd.to_datetime(
    gh_clean["cutoff_date"], errors="coerce", utc=True
).dt.tz_localize(None)

count_columns = [
    "push_events_h1_2024",
    "push_events_h2_2024",
    "push_events_2024",
    "issue_events_2024",
    "issue_comment_events_2024",
    "pull_request_events_2024",
    "review_events_2024",
    "release_events_2024",
    "stars_received_2024",
    "fork_events_2024",
    "distinct_actors_2024",
    "active_months_2024",
    "active_days_2024",
    "qualifying_events_h1_2024",
    "qualifying_events_h2_2024",
]
recency_columns = [
    "days_since_last_push_at_cutoff",
    "days_since_last_release_event_at_cutoff",
    "qualifying_event_change_h2_vs_h1",
]

conversion_audit = []

for column in count_columns + recency_columns + ["inactive_next_365d"]:
    originally_missing = gh_clean[column].isna()
    converted = pd.to_numeric(gh_clean[column], errors="coerce")
    parse_failed = gh_clean[column].notna() & converted.isna()

    conversion_audit.append({
        "column": column,
        "missing_before_conversion": int(originally_missing.sum()),
        "unparseable_values": int(parse_failed.sum()),
    })
    gh_clean[column] = converted

pd.DataFrame(conversion_audit).set_index("column")

if gh_clean[["repo_id", "repo_full_name", "cutoff_date"]].isna().any().any():
    raise ValueError("Missing repository identifier/name or invalid cutoff date found.")
if gh_clean.duplicated(["repo_id", "cutoff_date"]).any():
    raise ValueError("Conflicting duplicate repository/cutoff rows found; investigate before joining.")
if gh_clean["inactive_next_365d"].isna().any():
    raise ValueError("Target label is missing; do not impute labels.")
if not gh_clean["inactive_next_365d"].isin([0, 1]).all():
    raise ValueError("Target label must contain only 0 and 1.")
if (gh_clean[count_columns].dropna() < 0).any().any():
    raise ValueError("Negative event/count values found; investigate source data.")
if (gh_clean[recency_columns[:2]].dropna() < 0).any().any():
    raise ValueError("Negative recency values found; check cutoff/date logic.")

# Preserve meaningful missingness; add explicit flags before any model-time imputation.
missing_flag_columns = [
    "days_since_last_push_at_cutoff",
    "days_since_last_release_event_at_cutoff",
    "qualifying_event_change_h2_vs_h1",
]
for column in missing_flag_columns:
    gh_clean[f"{column}_missing"] = gh_clean[column].isna().astype("int8")

gh_clean["has_push_event_2024"] = gh_clean["push_events_2024"].gt(0).astype("int8")
gh_clean["has_release_event_2024"] = gh_clean["release_events_2024"].gt(0).astype("int8")
gh_clean["h1_qualifying_events_zero"] = gh_clean["qualifying_events_h1_2024"].eq(0).astype("int8")

assert gh_clean["days_since_last_push_at_cutoff_missing"].equals(
    gh_clean["push_events_2024"].eq(0).astype("int8")
)
assert gh_clean["days_since_last_release_event_at_cutoff_missing"].equals(
    gh_clean["release_events_2024"].eq(0).astype("int8")
)
assert gh_clean["qualifying_event_change_h2_vs_h1_missing"].equals(
    gh_clean["qualifying_events_h1_2024"].eq(0).astype("int8")
)

identifier_columns = ["repo_id", "repo_full_name", "cutoff_date"]
target_column = "inactive_next_365d"
feature_columns = [
    column for column in gh_clean.columns
    if column not in identifier_columns + [target_column]
]

print("Clean rows:", len(gh_clean))
print("Duplicate rows removed:", len(df) - len(gh_clean))
print("Label balance:")
print(gh_clean[target_column].value_counts(normalize=True).sort_index())
print("Missing feature values:")
print(gh_clean[feature_columns].isna().sum().sort_values(ascending=False).head(10))
print("Missing-value flag totals:")
print(gh_clean[[f"{column}_missing" for column in missing_flag_columns]].sum())


gh_clean.to_csv("../data/gh_archive_results_clean.csv", index=False)

**Notes:**

With the cleaned dataset, it still contains 62,627 rows but repository IDs were standardized as strings, repository names were lowerecased, and cutoff dates were pased as datatimes.The required identifiers was checked for missing or invalid inputs and missing values in the features `days_since_last_push_at_cutoff_missing`, `days_since_last_release_event_at_cutoff_missing`, and `qualifying_event_change_h2_vs_h1_missing` were retained and flagged as missing. This can mean that there was no matching 2024 event that was observed or it can mean that H2/H1 change classified as undefined because H1 qualifying events were zero. With that said, they should not be interpreted as ordinary zero values. 

## Visualize the Clean Dataset

The cleaned dataset has one cutoff date, so it does not support a longitudinal time-series plot. The two 2024 half-year columns can be compared as a limited before/after view; this is descriptive and covers only repositories with at least one qualifying event in 2024.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# The binary target is categorical; show both counts and proportions.
target_counts = gh_clean[target_column].value_counts().reindex([0, 1], fill_value=0)
target_labels = ["Active next 365 days", "Inactive next 365 days"]
target_colors = ["#2E7D68", "#C85A54"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(target_labels, target_counts.values, color=target_colors)
ax.set_title("Repository Outcome Balance")
ax.set_ylabel("Repositories")
ax.bar_label(
    bars,
    labels=[
        f"{count:,} ({count / len(gh_clean):.1%})"
        for count in target_counts.values
    ],
    padding=3,
)
ax.set_ylim(0, target_counts.max() * 1.15)
plt.xticks(rotation=10, ha="right")
plt.tight_layout()
plt.show()

### Bar Charts: Comparing Categorical Data

The target is the main categorical variable. The table and grouped bars below compare the prevalence of binary indicators between the two outcome groups. Percentages are calculated within each outcome group.

In [ ]:
binary_feature_columns = [
    "days_since_last_push_at_cutoff_missing",
    "days_since_last_release_event_at_cutoff_missing",
    "qualifying_event_change_h2_vs_h1_missing",
    "has_push_event_2024",
    "has_release_event_2024",
    "h1_qualifying_events_zero",
]

outcome_labels = {0: "Active", 1: "Inactive"}
binary_summary = pd.DataFrame(index=binary_feature_columns)
for outcome, label in outcome_labels.items():
    outcome_rows = gh_clean.loc[gh_clean[target_column].eq(outcome)]
    binary_summary[f"{label} (%)"] = [
        outcome_rows[column].eq(1).mean() * 100
        for column in binary_feature_columns
    ]

binary_summary.index.name = "Binary indicator"
binary_summary

In [ ]:
ax = binary_summary.plot(
    kind="barh",
    figsize=(9, 5),
    color=["#2E7D68", "#C85A54"],
    width=0.78,
)
ax.set_title("Binary Feature Prevalence by Repository Outcome")
ax.set_xlabel("Repositories where indicator is true (%)")
ax.set_ylabel("")
ax.legend(title="Outcome")
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

### Outlier Review

Upper percentiles are used to identify candidates for validation, not for automatic deletion. Large repository activity counts may be legitimate, but best to confirm a source-data or logic error before changing any row.

In [ ]:
outlier_summary = (
    gh_clean[count_columns + recency_columns]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .T[["count", "mean", "50%", "90%", "95%", "99%", "max"]]
    .round(2)
)
outlier_summary